In [1]:
from pbase.config.ray import init_ray
import shadow_price_prediction

# Initialize Ray for parallel processing
extra_modules = [shadow_price_prediction]
init_ray(address='ray://10.8.0.36:10001', extra_modules=extra_modules)

import pandas as pd
import numpy as np
import random
from pathlib import Path
from pbase.utils.ray import parallel_equal_pool
from shadow_price_prediction.config import PredictionConfig

# Import tuning utilities for resumable search
from shadow_price_prediction.tuning_utils import (
    load_previous_params,
    sample_params,
    run_single_experiment
)

pd.set_option('display.max_columns', 99)

config = PredictionConfig()

2025-12-04 20:35:26,583	INFO client_builder.py:242 -- Passing the following kwargs to ray.init() on the server: log_to_driver
2025-12-04 20:35:26,927	INFO packaging.py:588 -- Creating a file package for local module '/home/xyz/workspace/research-spice-shadow-price-pred/src/shadow_price_prediction'.
2025-12-04 20:35:26,932	INFO packaging.py:380 -- Pushing file package 'gcs://_ray_pkg_83b34365ae3573b7.zip' (0.35MiB) to Ray cluster...
2025-12-04 20:35:26,937	INFO packaging.py:393 -- Successfully pushed file package 'gcs://_ray_pkg_83b34365ae3573b7.zip'.
SIGTERM handler is not set because current thread is not the main thread.


In [2]:
# N_ITER = 10000  # Total number of experiments to run (including previous ones)
# BASE_OUTPUT_DIR = Path('/opt/temp/haoyan/param_search_new')

In [3]:
import pandas as pd
from shadow_price_prediction.config import PredictionConfig
from shadow_price_prediction.tuning_utils import update_config_with_params
from shadow_price_prediction.pipeline import ShadowPricePipeline

def create_pipeline_from_row(row):
    """
    Create a ShadowPricePipeline using the parameters from a single row of results.
    
    Args:
        row: DataFrame row with parameters
    
    Returns:
        ShadowPricePipeline configured with the parameters
    """
    params_dict = row.to_dict()
        
    print("Initializing pipeline with selected parameters:")
    for k, v in params_dict.items():
         print(f"  {k}: {v}")

    # Create Config
    config = PredictionConfig()
    
    # Update Config with Params
    config = update_config_with_params(config, params_dict)
    
    # Initialize Pipeline
    pipeline = ShadowPricePipeline(config)
    
    return pipeline

In [4]:
params = {
    'clf_xgb_n_estimators': 500,
    'clf_xgb_max_depth': 3,
    'clf_xgb_learning_rate': 0.3,
    'clf_xgb_gamma': 0,
    'clf_xgb_min_child_weight': 10,
    'clf_xgb_reg_alpha': 0.5,
    'clf_xgb_reg_lambda': 5,
    'reg_xgb_n_estimators': 500,
    'reg_xgb_max_depth': 3,
    'reg_xgb_learning_rate': 0.05,
    'reg_xgb_gamma': 0,
    'reg_xgb_min_child_weight': 10,
    'reg_xgb_reg_alpha': 0.5,
    'reg_xgb_reg_lambda': 10,
    'lr_penalty': 'l2',
    'lr_C': 10.0,
    'enet_alpha': 1.0,
    'enet_l1_ratio': 0.9,
    'threshold_beta': 50,
    'clf_xgb_thres': 1,
    'reg_xgb_thres': 1,
    'clf_lr_thres': 0.1,
    'reg_enet_thres': 0.1,

    'short_term_clf_xgb_w': 1,
    'short_term_reg_xgb_w': 1,
    'long_term_clf_xgb_w': 1,
    'long_term_reg_xgb_w': 1,
}

pipeline = create_pipeline_from_row(pd.Series(params))

Initializing pipeline with selected parameters:
  clf_xgb_n_estimators: 500
  clf_xgb_max_depth: 3
  clf_xgb_learning_rate: 0.3
  clf_xgb_gamma: 0
  clf_xgb_min_child_weight: 10
  clf_xgb_reg_alpha: 0.5
  clf_xgb_reg_lambda: 5
  reg_xgb_n_estimators: 500
  reg_xgb_max_depth: 3
  reg_xgb_learning_rate: 0.05
  reg_xgb_gamma: 0
  reg_xgb_min_child_weight: 10
  reg_xgb_reg_alpha: 0.5
  reg_xgb_reg_lambda: 10
  lr_penalty: l2
  lr_C: 10.0
  enet_alpha: 1.0
  enet_l1_ratio: 0.9
  threshold_beta: 50
  clf_xgb_thres: 1
  reg_xgb_thres: 1
  clf_lr_thres: 0.1
  reg_enet_thres: 0.1
  short_term_clf_xgb_w: 1
  short_term_reg_xgb_w: 1
  long_term_clf_xgb_w: 1
  long_term_reg_xgb_w: 1


In [5]:
test_periods = []

def get_auction_period_type(auction_month, market_month):
    q_period = f'q{(market_month.month - 6) % 12 // 3 + 1}'
    f_period = f'f{(market_month.year - auction_month.year)*12 + market_month.month - auction_month.month}'
    period_type = None
    # all_periods = aptools.tools._rto_period_type_class.get_valid_periods(auction_month)
    all_periods = config.iso.auction_schedule.get(auction_month.month, [])
    if q_period in all_periods:
        period_type = q_period
    elif f_period in all_periods:
        period_type = f_period
    return period_type

for auction_month in pd.date_range('2025-10', '2025-10', freq='MS'):
    market_st = auction_month
    # market_et = market_st
    market_et = market_st + pd.offsets.MonthBegin(12) if market_st.month >= 6 else market_st
    market_et = market_et.replace(month=5)
    
    for market_month in pd.date_range(start=market_st, end=market_et, freq='MS'):
        if get_auction_period_type(auction_month, market_month) is None:
            continue
        test_periods.append((auction_month, market_month))
len(test_periods)

8

In [6]:
results_per_outage, final_results, metrics, training_res, models = pipeline.run(
    test_periods=test_periods,
    # test_periods=test_periods[2:3],
    verbose=True,
    use_parallel=True,
    n_jobs=36,
    output_dir='/opt/temp/haoyan/spice_ml1/',
    # output_dir=None,
    refresh=True,
    class_type='onpeak',
    # branch_name='XRDSGRANC11_1  1',
    branch_name=None,
)

SHADOW PRICE PREDICTION PIPELINE - PARALLEL PER-AUCTION-MONTH MODE
Total Test Periods: 8
Unique Auction Months: 8

  Auction: 2025-10 → 1 market month(s)
    - Market: 2025-10
  Auction: 2025-10 → 1 market month(s)
    - Market: 2025-11
  Auction: 2025-10 → 1 market month(s)
    - Market: 2025-12
  Auction: 2025-10 → 1 market month(s)
    - Market: 2026-01
  Auction: 2025-10 → 1 market month(s)
    - Market: 2026-02
  Auction: 2025-10 → 1 market month(s)
    - Market: 2026-03
  Auction: 2025-10 → 1 market month(s)
    - Market: 2026-04
  Auction: 2025-10 → 1 market month(s)
    - Market: 2026-05
Class Type: onpeak

🚀 Using Ray parallel processing for 8 auction months
   Workers: 36

[PARALLEL PROCESSING: Training and Predicting for All Auction Months]


  0%|          | 0/8 [00:00<?, ?it/s]

(PoolActor pid=2568094, ip=10.42.8.15) 25-12-04 20:35:31 EST | INFO    | dotenv.main          | _get_stream         | #61  | python-dotenv could not find configuration file .env.
(PoolActor pid=2568094, ip=10.42.8.15) 25-12-04 20:35:31 EST | WARNING | pbase.utils.utils    | load_env            | #478 | .env could not be found...
(PoolActor pid=2568094, ip=10.42.8.15) 25-12-04 20:35:31 EST | INFO    | pbase.config.base    | load_global_config  | #435 | Loading config from local cached json file...
(PoolActor pid=2568094, ip=10.42.8.15) 25-12-04 20:35:31 EST | INFO    | pbase.config.base    | __init__            | #44  | Init global config...
(PoolActor pid=2815899, ip=10.42.6.149) 25-12-04 20:35:31 EST | INFO    | dotenv.main          | _get_stream         | #61  | python-dotenv could not find configuration file .env.
(PoolActor pid=2815899, ip=10.42.6.149) 25-12-04 20:35:31 EST | WARNING | pbase.utils.utils    | load_env            | #478 | .env could not be found...
(PoolActor pid=281

(PoolActor pid=2568094, ip=10.42.8.15) 
(PoolActor pid=2568094, ip=10.42.8.15) ================================================================================
(PoolActor pid=2568094, ip=10.42.8.15) [Processing Auction Month: 2025-10]
(PoolActor pid=2568094, ip=10.42.8.15)   Market Months: 1
(PoolActor pid=2568094, ip=10.42.8.15)     - 2026-03
(PoolActor pid=2568094, ip=10.42.8.15) ================================================================================
(PoolActor pid=2579930, ip=10.42.4.173) 
(PoolActor pid=2579930, ip=10.42.4.173) ================================================================================
(PoolActor pid=2579930, ip=10.42.4.173) [Processing Auction Month: 2025-10]
(PoolActor pid=2579930, ip=10.42.4.173)   Market Months: 1
(PoolActor pid=2579930, ip=10.42.4.173)     - 2025-12
(PoolActor pid=2579930, ip=10.42.4.173) ================================================================================
(PoolActor pid=2504802, ip=10.42.7.70) 
(PoolActor pid=2504802

In [14]:
a = pd.read_parquet('/opt/temp/haoyan/spice_ml1/auction_month=2025-10/market_month=2026-05/class_type=onpeak/final_results.parquet')
b = pd.read_parquet('/opt/temp/haoyan/spice_ml/auction_month=2025-10/market_month=2026-05/class_type=onpeak/final_results.parquet')
(a['binding_probability_scaled'] - b['binding_probability_scaled']).abs().sum()

np.float64(0.0)

In [15]:
# PJM Execution
from shadow_price_prediction.iso_configs import PJM_ISO_CONFIG, PJM_HORIZON_GROUPS

# 1. Update Config for PJM
config = PredictionConfig()
config.iso = PJM_ISO_CONFIG
config.horizon_groups = PJM_HORIZON_GROUPS

# 2. Initialize Pipeline
# Using the same params as MISO for now, or you can define new ones
pipeline = ShadowPricePipeline(config)

# 3. Define Test Periods for PJM
test_periods = []

# PJM doesn't use q-periods, so logic is simpler
# Just check if f-period is in schedule
def get_pjm_period_type(auction_month, market_month):
    f_period = f'f{(market_month.year - auction_month.year)*12 + market_month.month - auction_month.month}'
    
    all_periods = config.iso.auction_schedule.get(auction_month.month, [])
    if f_period in all_periods:
        return f_period
    return None

# Example range for PJM
for auction_month in pd.date_range('2025-12', '2025-12', freq='MS'):
    # PJM usually goes out 3 years, but let's just do 1 year for testing
    market_st = auction_month
    market_et = market_st + pd.offsets.MonthBegin(12)
    
    for market_month in pd.date_range(start=market_st, end=market_et, freq='MS'):
        if get_pjm_period_type(auction_month, market_month) is None:
            continue
        test_periods.append((auction_month, market_month))

print(f"PJM Test Periods: {len(test_periods)}")

PJM Test Periods: 6


In [16]:
# 4. Run Pipeline
results_per_outage, final_results, metrics, training_res, models = pipeline.run(
    test_periods=test_periods,
    verbose=True,
    use_parallel=True,
    n_jobs=36,
    output_dir='/opt/temp/haoyan/spice_ml_pjm/', # Separate output dir for PJM
    refresh=True,
    class_type='onpeak',
    branch_name=None,
)

SHADOW PRICE PREDICTION PIPELINE - PARALLEL PER-AUCTION-MONTH MODE
Total Test Periods: 6
Unique Auction Months: 6

  Auction: 2025-12 → 1 market month(s)
    - Market: 2025-12
  Auction: 2025-12 → 1 market month(s)
    - Market: 2026-01
  Auction: 2025-12 → 1 market month(s)
    - Market: 2026-02
  Auction: 2025-12 → 1 market month(s)
    - Market: 2026-03
  Auction: 2025-12 → 1 market month(s)
    - Market: 2026-04
  Auction: 2025-12 → 1 market month(s)
    - Market: 2026-05
Class Type: onpeak

🚀 Using Ray parallel processing for 6 auction months
   Workers: 36

[PARALLEL PROCESSING: Training and Predicting for All Auction Months]


  0%|          | 0/6 [00:00<?, ?it/s]

(PoolActor pid=2580040, ip=10.42.4.173) 
(PoolActor pid=2580040, ip=10.42.4.173) ================================================================================
(PoolActor pid=2580040, ip=10.42.4.173) [Processing Auction Month: 2025-12]
(PoolActor pid=2580040, ip=10.42.4.173)   Market Months: 1
(PoolActor pid=2580040, ip=10.42.4.173)     - 2025-12
(PoolActor pid=2580040, ip=10.42.4.173) ================================================================================
(PoolActor pid=2507600, ip=10.42.0.182) 
(PoolActor pid=2507600, ip=10.42.0.182) ================================================================================
(PoolActor pid=2507600, ip=10.42.0.182) [Processing Auction Month: 2025-12]
(PoolActor pid=2507600, ip=10.42.0.182)   Market Months: 1
(PoolActor pid=2507600, ip=10.42.0.182)     - 2026-04
(PoolActor pid=2507600, ip=10.42.0.182) ================================================================================
(PoolActor pid=2505016, ip=10.42.7.70) 
(PoolActor pid=2

(PoolActor pid=2568205, ip=10.42.8.15) 25-12-04 20:38:16 EST | INFO    | dotenv.main          | _get_stream         | #61  | python-dotenv could not find configuration file .env.
(PoolActor pid=2568205, ip=10.42.8.15) 25-12-04 20:38:16 EST | WARNING | pbase.utils.utils    | load_env            | #478 | .env could not be found...
(PoolActor pid=2568205, ip=10.42.8.15) 25-12-04 20:38:16 EST | INFO    | pbase.config.base    | load_global_config  | #435 | Loading config from local cached json file...
(PoolActor pid=2568205, ip=10.42.8.15) 25-12-04 20:38:16 EST | INFO    | pbase.config.base    | __init__            | #44  | Init global config...
(PoolActor pid=2580040, ip=10.42.4.173) 25-12-04 20:38:16 EST | INFO    | dotenv.main          | _get_stream         | #61  | python-dotenv could not find configuration file .env.
(PoolActor pid=2580040, ip=10.42.4.173) 25-12-04 20:38:16 EST | WARNING | pbase.utils.utils    | load_env            | #478 | .env could not be found...
(PoolActor pid=258

(PoolActor pid=2568205, ip=10.42.8.15) 
(PoolActor pid=2568205, ip=10.42.8.15) [STEP 1: Training Period Calculation]
(PoolActor pid=2568205, ip=10.42.8.15)   Training Range: 2024-12 to 2025-11
(PoolActor pid=2568205, ip=10.42.8.15) 
(PoolActor pid=2568205, ip=10.42.8.15) [STEP 2: Loading Training Data]
(PoolActor pid=2568205, ip=10.42.8.15)   Loading training data for 2024-12 (Types: ['f3', 'f4', 'f5'])...
(PoolActor pid=2816010, ip=10.42.6.149) 
(PoolActor pid=2816010, ip=10.42.6.149) [STEP 1: Training Period Calculation]
(PoolActor pid=2816010, ip=10.42.6.149)   Training Range: 2024-12 to 2025-11
(PoolActor pid=2816010, ip=10.42.6.149) 
(PoolActor pid=2816010, ip=10.42.6.149) [STEP 2: Loading Training Data]
(PoolActor pid=2816010, ip=10.42.6.149)   Loading training data for 2024-12 (Types: ['f1'])...
(PoolActor pid=2580040, ip=10.42.4.173) 
(PoolActor pid=2580040, ip=10.42.4.173) [STEP 1: Training Period Calculation]
(PoolActor pid=2580040, ip=10.42.4.173)   Training Range: 2024-12 t

...

(PoolActor pid=2816010, ip=10.42.6.149)   Found 3639 unique branch-flow pairs in test set.
(PoolActor pid=2816010, ip=10.42.6.149) 
(PoolActor pid=2816010, ip=10.42.6.149) [STEP 4: Training Models for Auction Month 2025-12]
(PoolActor pid=2816010, ip=10.42.6.149) 
(PoolActor pid=2816010, ip=10.42.6.149) [Training Classification Models + Threshold Optimization]
(PoolActor pid=2816010, ip=10.42.6.149) --------------------------------------------------------------------------------
(PoolActor pid=2816010, ip=10.42.6.149)   Skipping groups not in test periods: f0, f2, long
(PoolActor pid=2816010, ip=10.42.6.149) 
(PoolActor pid=2816010, ip=10.42.6.149) Processing F1 TERM models...
(PoolActor pid=2816010, ip=10.42.6.149)   Training default fallback classifier (f1)...


(PoolActor pid=2580040, ip=10.42.4.173) 25-12-04 20:38:32 EST | INFO    | pbase.analysis.tools.base | add_offpeak         | #7935 | extended offpeak: 768 hours => 1168 hours


(PoolActor pid=2566623, ip=10.42.9.131)     ✓ Optimal threshold: 0.964 (F-beta=0.524)
(PoolActor pid=2566623, ip=10.42.9.131)   Training branch-specific classifiers (f2)...
(PoolActor pid=2580040, ip=10.42.4.173)   Applying label modification based on prob_exceed_80...
(PoolActor pid=2580040, ip=10.42.4.173)     Modified 708924 labels to 0 where prob_exceed_80 is 0.
(PoolActor pid=2580040, ip=10.42.4.173) 
(PoolActor pid=2580040, ip=10.42.4.173) [STEP 3: Identifying Target Branches]
(PoolActor pid=2580040, ip=10.42.4.173)   Identifying target branches from test periods...


(PoolActor pid=2580040, ip=10.42.4.173) 25-12-04 20:38:33 EST | INFO    | pbase.analysis.tools.base | add_offpeak         | #7935 | extended offpeak: 745 hours => 1170 hours
(PoolActor pid=2580040, ip=10.42.4.173) 25-12-04 20:38:33 EST | INFO    | pbase.analysis.tools.base | add_offpeak         | #7935 | extended offpeak: 768 hours => 1168 hours
(PoolActor pid=2580040, ip=10.42.4.173) 25-12-04 20:38:33 EST | INFO    | pbase.analysis.tools.base | add_offpeak         | #7935 | extended offpeak: 744 hours => 1136 hours
(PoolActor pid=2580040, ip=10.42.4.173) 25-12-04 20:38:33 EST | INFO    | pbase.analysis.tools.base | add_offpeak         | #7935 | extended offpeak: 696 hours => 1072 hours
(PoolActor pid=2580040, ip=10.42.4.173) 25-12-04 20:38:33 EST | INFO    | pbase.analysis.tools.base | add_offpeak         | #7935 | extended offpeak: 768 hours => 1184 hours
(PoolActor pid=2580040, ip=10.42.4.173) 25-12-04 20:38:33 EST | INFO    | pbase.analysis.tools.base | add_offpeak         | #7935 

(PoolActor pid=2505016, ip=10.42.7.70)     ✓ Default ensemble trained (1 models)
(PoolActor pid=2505016, ip=10.42.7.70)     Total samples: 1,048,088
(PoolActor pid=2505016, ip=10.42.7.70)   Optimizing threshold for default ensemble (long)...
(PoolActor pid=2568205, ip=10.42.8.15)     ✓ Trained 297 branch models (skipped 3193)
(PoolActor pid=2568205, ip=10.42.8.15) 
(PoolActor pid=2568205, ip=10.42.8.15) ✓ Classification Training Complete
(PoolActor pid=2568205, ip=10.42.8.15) 
(PoolActor pid=2568205, ip=10.42.8.15) [STEP 5: Training Regressors for Auction Month 2025-12]
(PoolActor pid=2580040, ip=10.42.4.173)   Found 3769 unique branch-flow pairs in test set.
(PoolActor pid=2580040, ip=10.42.4.173) 
(PoolActor pid=2580040, ip=10.42.4.173) [STEP 4: Training Models for Auction Month 2025-12]
(PoolActor pid=2580040, ip=10.42.4.173) 
(PoolActor pid=2580040, ip=10.42.4.173) [Training Classification Models + Threshold Optimization]
(PoolActor pid=2580040, ip=10.42.4.173) --------------------